In [1]:
from pathlib import Path
import os

# We start from the notebook location and walk upwards until we find the "AutoML" folder
here = Path.cwd()
project_root = None

for p in [here] + list(here.parents):
    if (p / "AutoML").exists():
        project_root = p / "AutoML"
        break

if project_root is None:
    raise FileNotFoundError("Could not find the AutoML folder above this notebook. Check where the notebook is saved.")

OUTPUTS_DIR = project_root / "outputs"
TEST_FRAMES_DIR = OUTPUTS_DIR / "test_frames"
MODELS_SAVED_DIR = OUTPUTS_DIR / "models_saved"
MODELS_DIR = OUTPUTS_DIR / "models"
METRICS_DIR = OUTPUTS_DIR / "metrics"
PLOTS_DIR = OUTPUTS_DIR / "plots"

print("Project root:", project_root)
print("OUTPUTS_DIR exists?", OUTPUTS_DIR.exists(), OUTPUTS_DIR)
print("TEST_FRAMES_DIR exists?", TEST_FRAMES_DIR.exists(), TEST_FRAMES_DIR)
print("MODELS_SAVED_DIR exists?", MODELS_SAVED_DIR.exists(), MODELS_SAVED_DIR)
print("MODELS_DIR exists?", MODELS_DIR.exists(), MODELS_DIR)
print("METRICS_DIR exists?", METRICS_DIR.exists(), METRICS_DIR)
print("PLOTS_DIR exists?", PLOTS_DIR.exists(), PLOTS_DIR)

Project root: C:\Users\sohib\Documents\Final Year Project\AutoML
OUTPUTS_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs
TEST_FRAMES_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\test_frames
MODELS_SAVED_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\models_saved
MODELS_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\models
METRICS_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\metrics
PLOTS_DIR exists? True C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\plots


In [2]:
def list_dir(d: Path, title: str):
    print("\n" + title)
    if not d.exists():
        print("  (missing)")
        return
    items = sorted([x.name for x in d.iterdir()])
    if not items:
        print("  (empty)")
        return
    for x in items[:50]:
        print("  -", x)
    if len(items) > 50:
        print(f"  ... and {len(items)-50} more")

list_dir(OUTPUTS_DIR, "outputs/")
list_dir(TEST_FRAMES_DIR, "outputs/test_frames/")
list_dir(MODELS_SAVED_DIR, "outputs/models_saved/")
list_dir(MODELS_DIR, "outputs/models/")
list_dir(METRICS_DIR, "outputs/metrics/")
list_dir(PLOTS_DIR, "outputs/plots/")


outputs/
  - features
  - leaderboards
  - metrics
  - models
  - models_saved
  - plots
  - test_frames

outputs/test_frames/
  - test_ae.parquet
  - test_baseline.parquet
  - test_kdd.parquet
  - test_pca.parquet
  - test_vae.parquet

outputs/models_saved/
  - GBM_1_AutoML_1_20260213_181254
  - GBM_4_AutoML_3_20260213_200153
  - GBM_grid_1_AutoML_2_20260213_185958_model_3
  - GBM_grid_1_AutoML_4_20260213_205118_model_3
  - GBM_grid_1_AutoML_5_20260213_220344_model_5
  - autoencoder
  - vae

outputs/models/
  - GBM_1_AutoML_1_20260213_181254
  - kdd_preprocessor.joblib
  - pca_95.joblib
  - preprocessor.joblib

outputs/metrics/
  - balancing_info.json
  - candidates_all.csv
  - cbme_ranked_all.csv
  - cbme_stage1_survivors.csv
  - cbme_stage2_finalists.csv
  - cbme_winner.json
  - comparison_baseline_vs_pca.csv
  - confusion_kdd_baseline.json
  - confusion_unsw_baseline.json
  - final_comparison_table.csv
  - kdd_balancing_info.json
  - kdd_test_metrics_bestF1.json
  - master_results

In [3]:
import h2o
h2o.init()
print("Connected to:", h2o.connection().base_url)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,27 mins 10 secs
H2O_cluster_timezone:,Europe/London
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,3 months and 7 days
H2O_cluster_name:,H2O_from_python_sohib_suaek0
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.890 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Connected to: http://localhost:54321


In [4]:
print("H2O models currently in memory:")
print(h2o.ls())

H2O models currently in memory:


C:\Users\sohib\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


                                                 key
0                     GBM_1_AutoML_1_20260213_181254
1  modelmetrics_GBM_1_AutoML_1_20260213_181254@-3...
2  modelmetrics_GBM_1_AutoML_1_20260213_181254@-3...


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import h2o

# pick your label column name (set this correctly)
# common options: "label", "Label", "attack", "class", "is_attack"
LABEL_COL = "label"   # <-- CHANGE if your dataset uses a different label column

test_files = {
    "UNSW_baseline": TEST_FRAMES_DIR / "test_baseline.parquet",
    "UNSW_pca":      TEST_FRAMES_DIR / "test_pca.parquet",
    "UNSW_ae":       TEST_FRAMES_DIR / "test_ae.parquet",
    "UNSW_vae":      TEST_FRAMES_DIR / "test_vae.parquet",
    "NSL_KDD":       TEST_FRAMES_DIR / "test_kdd.parquet",
}

dfs = {}
h2_frames = {}

for name, path in test_files.items():
    df = pd.read_parquet(path)
    dfs[name] = df
    if LABEL_COL not in df.columns:
        raise ValueError(f"{name}: LABEL_COL='{LABEL_COL}' not found. Columns include: {list(df.columns)[:20]} ...")
    hf = h2o.H2OFrame(df)
    # make sure label is categorical for binomial metrics
    hf[LABEL_COL] = hf[LABEL_COL].asfactor()
    h2_frames[name] = hf
    print(name, "shape:", df.shape)

print("\nLoaded frames:", list(h2_frames.keys()))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
UNSW_baseline shape: (82332, 98)
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
UNSW_pca shape: (82332, 23)
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
UNSW_ae shape: (82332, 49)
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
UNSW_vae shape: (82332, 25)
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
NSL_KDD shape: (22544, 62)

Loaded frames: ['UNSW_baseline', 'UNSW_pca', 'UNSW_ae', 'UNSW_vae', 'NSL_KDD']


In [6]:
import h2o
model_id = "GBM_1_AutoML_1_20260213_181254"
model = h2o.get_model(model_id)
print("Loaded model:", model.model_id)

Loaded model: GBM_1_AutoML_1_20260213_181254


In [14]:
import numpy as np

def _cm_to_counts(cm_obj):
    """
    Convert H2O ConfusionMatrix object -> (TN, FP, FN, TP)
    Works with typical binomial confusion matrix layout.
    """
    # ConfusionMatrix has a 'table' (H2OTwoDimTable)
    tbl = cm_obj.table
    df = tbl.as_data_frame()

    # Keep numeric cells only
    numeric = df.select_dtypes(include=[np.number]).values

    # Expected numeric layout often like:
    # row0: [TN, FP, Error]
    # row1: [FN, TP, Error]
    # We'll take first two columns.
    TN = float(numeric[0][0])
    FP = float(numeric[0][1])
    FN = float(numeric[1][0])
    TP = float(numeric[1][1])
    return TN, FP, FN, TP


def eval_best_f1(model, hf):
    perf = model.model_performance(hf)

    # 1) best F1 threshold
    thr = float(perf.find_threshold_by_max_metric("f1"))

    # 2) confusion matrix at that threshold (returns LIST in your version)
    cm_list = perf.confusion_matrix(metrics="f1", thresholds=[thr])
    cm_obj = cm_list[0]  # take first ConfusionMatrix object

    TN, FP, FN, TP = _cm_to_counts(cm_obj)

    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = (2*precision*recall)/(precision+recall) if (precision+recall) else 0.0
    fpr       = FP / (FP + TN) if (FP + TN) else 0.0

    return {
        "AUC": float(perf.auc()),
        "AUCPR": float(perf.aucpr()),
        "threshold_bestF1": thr,
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "FPR": float(fpr),
        "TN": TN, "FP": FP, "FN": FN, "TP": TP
    }

### Reproducibility Verification

The final trained model was reloaded from disk and evaluated using the saved test datasets.  
The reproduced evaluation metrics match the results obtained during the original experimental pipeline, confirming that the AutoML training process, model persistence, and evaluation procedures are reproducible.

This validation ensures that the experimental results reported in the study can be independently reproduced.

In [17]:
datasets_to_test = ["UNSW_baseline", "NSL_KDD"]

results = []

for name in datasets_to_test:
    hf = h2_frames[name]
    m = eval_best_f1(model, hf)
    results.append({"Dataset": name, **m})

res_df = pd.DataFrame(results)
res_df

,Dataset,AUC,AUCPR,threshold_bestF1,Precision,Recall,F1,FPR,TN,FP,FN,TP
0,UNSW_baseline,0.984602,0.988654,0.867532,0.954311,0.919218,0.936436,0.053919,35005.0,1995.0,3662.0,41670.0
1,NSL_KDD,0.202396,0.404975,0.966298,0.569242,1.000000,0.725500,1.000000,0.0,9711.0,0.0,12833.0


### Cross-Dataset Reproducibility Note

The NSL-KDD dataset requires a dedicated preprocessing pipeline
(`kdd_preprocessor.joblib`) to align its features with the UNSW-NB15
training representation. In this reproducibility notebook the raw NSL
test frame is evaluated without applying this preprocessing step,
which results in degraded performance metrics.

The correct NSL-KDD evaluation results reported in the dissertation
are obtained using the full preprocessing pipeline in the main
experimental notebook.

In [18]:
out_path = METRICS_DIR / "reproducibility_results.csv"
res_df.to_csv(out_path, index=False)

print("Reproducibility results saved to:", out_path)

Reproducibility results saved to: C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\metrics\reproducibility_results.csv
